In [ ]:
!pip install requests azure-storage-blob tqdm

In [ ]:
"""
street_view_metadata_accidents.py

Reads a CSV with latitude/longitude columns.
For each row, calls Google Street View metadata API:
    https://maps.googleapis.com/maps/api/streetview/metadata

If metadata.status == "OK", saves one TXT file per row using:
    <row_idx>.txt

Uploads the TXT files to Azure Blob Storage under:
    <container>/<prefix>/AccidentsMetadata/

Also stores a checkpoint so the script can resume safely in Colab.

Requirements:
    pip install requests azure-storage-blob tqdm
"""

# If running in Colab, install dependencies in a notebook cell first:
# !pip install requests azure-storage-blob tqdm

import os
import csv
import json
import time
import logging
from typing import List, Tuple, Dict, Any
from urllib.parse import urlencode

import requests
from tqdm import tqdm
from azure.storage.blob import BlobServiceClient, ContentSettings

# -----------------------
# USER CONFIG
# -----------------------
street_view_api_key = os.environ.get("STREET_VIEW_API_KEY", "")
azure_api_key = os.environ.get("AZURE_STORAGE_CONN_STR", "")

blob_path = ".../london/Accidents"   # format: "<container>/<optional/prefix>"
locations_csv = "all_59k_accidents_raw.csv"

radius = 50
source = "outdoor"
target_count = 60000                  # number of rows with metadata OK to collect
request_timeout = 30
max_retries = 3

pitch = 0

user_agent = "sv-accidents-metadata/1.0"

checkpoint_local = "accidents_metadata_checkpoint.json"
local_meta_folder = "Metadata"
remote_meta_folder = "Metadata"

# -----------------------
# logging
# -----------------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s")
logger = logging.getLogger("accidents_metadata")

# -----------------------
# CSV loader
# -----------------------
def load_locations_from_csv(path: str) -> List[Tuple[float, float]]:
    locs: List[Tuple[float, float]] = []

    with open(path, newline="", encoding="utf-8") as fh:
        reader = csv.DictReader(fh)
        for row in reader:
            try:
                lat = float(row["latitude"])
                lng = float(row["longitude"])
                locs.append((lat, lng))
            except Exception:
                continue

    return locs

# -----------------------
# Azure helpers
# -----------------------
def make_blob_service_client(azure_key: str) -> BlobServiceClient:
    if not azure_key:
        raise ValueError("AZURE_STORAGE_CONN_STR is missing.")

    if azure_key.startswith("DefaultEndpointsProtocol=") or "AccountName=" in azure_key:
        return BlobServiceClient.from_connection_string(azure_key)

    if azure_key.startswith("https://") and "sv=" in azure_key:
        account_url = azure_key.split("?", 1)[0]
        sas_token = azure_key.split("?", 1)[1]
        return BlobServiceClient(account_url=account_url, credential=sas_token)

    if ":" in azure_key and len(azure_key.split(":", 1)[1]) > 8:
        account, key = azure_key.split(":", 1)
        conn_str = (
            f"DefaultEndpointsProtocol=https;AccountName={account};"
            f"AccountKey={key};EndpointSuffix=core.windows.net"
        )
        return BlobServiceClient.from_connection_string(conn_str)

    raise ValueError(
        "Unrecognized AZURE_STORAGE_CONN_STR format. "
        "Use a connection string, SAS URL, or account:account_key."
    )

def parse_blob_path(blob_path_value: str):
    container_name, *rest = blob_path_value.split("/", 1)
    prefix = rest[0].rstrip("/") if rest else ""
    return container_name, prefix

blob_service = make_blob_service_client(azure_api_key)
container_name, prefix = parse_blob_path(blob_path)
container_client = blob_service.get_container_client(container_name)

try:
    container_client.create_container()
    logger.info("Created container: %s", container_name)
except Exception:
    logger.debug("Container already exists or cannot be created; continuing.")

def upload_text_to_blob(text: str, blob_rel_path: str, content_type: str = "text/plain"):
    blob_client = container_client.get_blob_client(blob_rel_path)
    blob_client.upload_blob(
        text.encode("utf-8"),
        overwrite=True,
        content_settings=ContentSettings(content_type=content_type)
    )

# -----------------------
# Google Street View metadata
# -----------------------
METADATA_BASE = "https://maps.googleapis.com/maps/api/streetview/metadata"

def build_metadata_url(lat: float, lng: float, radius_value: int = None, source_value: str = "outdoor"):
    r = int(radius_value) if radius_value is not None else int(radius)
    params = {
        "location": f"{lat},{lng}",
        "radius": r,
        "key": street_view_api_key,
        "source": source_value,
    }
    return f"{METADATA_BASE}?{urlencode(params)}"

def fetch_metadata(lat: float, lng: float, radius_value: int = None, retries: int = max_retries) -> Dict[str, Any]:
    if not street_view_api_key:
        raise ValueError("STREET_VIEW_API_KEY is missing.")

    url = build_metadata_url(lat, lng, radius_value=radius_value, source_value=source)
    headers = {"User-Agent": user_agent}
    last_exc = None

    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, headers=headers, timeout=request_timeout)
            if r.status_code == 200:
                return r.json()
            last_exc = Exception(f"HTTP {r.status_code}: {r.text[:300]}")
        except Exception as e:
            last_exc = e

        time.sleep(1.5 ** attempt)

    raise last_exc

# -----------------------
# Checkpoint management
# -----------------------
def load_checkpoint():
    if os.path.exists(checkpoint_local):
        try:
            with open(checkpoint_local, "r", encoding="utf-8") as fh:
                return json.load(fh)
        except Exception:
            logger.exception("Failed reading checkpoint; starting fresh.")
    return {
        "done_rows": {},   # row_idx -> info
        "saved_count": 0
    }

def save_checkpoint(cp):
    with open(checkpoint_local, "w", encoding="utf-8") as fh:
        json.dump(cp, fh, indent=2)

    # Best-effort upload of checkpoint to Azure
    try:
        blob_rel = f"{prefix}/checkpoint/{checkpoint_local}" if prefix else f"checkpoint/{checkpoint_local}"
        upload_text_to_blob(json.dumps(cp, indent=2), blob_rel, content_type="application/json")
    except Exception:
        logger.exception("Failed to upload checkpoint to Azure (continuing).")

# -----------------------
# TXT payload builder
# -----------------------
def make_txt_payload(
    row_idx: int,
    lat: float,
    lng: float,
    md: Dict[str, Any]
) -> str:
    payload = {
        "row_idx": row_idx,
        "pitch": pitch,
        "lat": lat,
        "lng": lng,
        "radius": radius,
        "source": source,
        "status": md.get("status"),
        "pano_id": md.get("pano_id"),
        "pano_date": md.get("date"),
        "location": md.get("location"),
        "copyright": md.get("copyright"),
        "metadata_response": md,
        "saved_at_unix": time.time(),
    }
    return json.dumps(payload, indent=2, ensure_ascii=False)

# -----------------------
# Main
# -----------------------
def main():
    os.makedirs(local_meta_folder, exist_ok=True)

    locs = load_locations_from_csv(locations_csv)
    total_points = len(locs)
    if total_points == 0:
        raise SystemExit("No locations loaded from CSV.")

    logger.info("Loaded %d points from CSV.", total_points)

    cp = load_checkpoint()
    done_rows = cp.get("done_rows", {})
    saved_count = int(cp.get("saved_count", 0))

    logger.info("Checkpoint says %d rows already finished.", len(done_rows))

    pbar = tqdm(total=target_count, initial=saved_count, desc="metadata saved")

    for row_idx, (lat, lng) in enumerate(locs):
        if str(row_idx) in done_rows:
            continue

        if saved_count >= target_count:
            break

        try:
            md = fetch_metadata(lat, lng, radius_value=radius)
        except Exception as e:
            logger.debug("Fetch failed for row %d: %s", row_idx, e)
            continue

        if md.get("status") != "OK":
            # Mark row as done so resume does not keep retrying rows with no Street View coverage.
            done_rows[str(row_idx)] = {
                "status": md.get("status"),
                "lat": lat,
                "lng": lng,
                "saved_files": []
            }
            cp["done_rows"] = done_rows
            save_checkpoint(cp)
            continue

        saved_files = []


        filename = f"{row_idx}.txt"
        local_path = os.path.join(local_meta_folder, filename)
        blob_rel = f"{prefix}/{remote_meta_folder}/{filename}" if prefix else f"{remote_meta_folder}/{filename}"

        txt = make_txt_payload(row_idx=row_idx, lat=lat, lng=lng, md=md)

        # Save locally
        with open(local_path, "w", encoding="utf-8") as fh:
            fh.write(txt)

        # Save to Azure
        upload_text_to_blob(txt, blob_rel, content_type="text/plain")
        saved_files.append(blob_rel)

        saved_count += 1
        cp["saved_count"] = saved_count
        done_rows[str(row_idx)] = {
            "status": "OK",
            "lat": lat,
            "lng": lng,
            "saved_files": saved_files,
            "pano_id": md.get("pano_id")
        }
        cp["done_rows"] = done_rows
        save_checkpoint(cp)

        pbar.update(1)

    pbar.close()
    logger.info("Finished. Saved %d rows with metadata OK.", saved_count)

if __name__ == "__main__":
    main()

In [ ]:
"""
street_view_images_accidents.py

Reads metadata TXT/JSON files from Azure Blob Storage under:
    <container>/<prefix>/Metadata/

For each metadata file, extracts pano_id and downloads 4 Street View images
at headings: 0, 90, 180, 270.

Saves images locally as:
    <row_idx>_<angle>.jpg

Uploads images to Azure Blob Storage under:
    <container>/<prefix>/Images/

Checkpoint is stored locally so execution can resume in Colab.

Requirements:
    pip install requests azure-storage-blob tqdm
"""

# In Colab, install first:
# !pip install requests azure-storage-blob tqdm

import os
import json
import time
import logging
from typing import Dict, Any, List

import requests
from tqdm import tqdm
from azure.storage.blob import BlobServiceClient, ContentSettings

# -----------------------
# USER CONFIG
# -----------------------
street_view_api_key = os.environ.get("STREET_VIEW_API_KEY", "")
azure_api_key = os.environ.get("AZURE_STORAGE_CONN_STR", "")

blob_path = "london/Accidents"

metadata_folder = "Metadata"
images_folder = "Images"
angles = [0, 90, 180, 270]
pitch = 0
image_size = "640x640"

request_timeout = 30
max_retries = 3
user_agent = "sv-accidents-images/1.0"

checkpoint_local = "accidents_images_checkpoint.json"
local_image_folder = "Images"
pano_cache_folder = "PanoCache"

# -----------------------
# logging
# -----------------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s")
logger = logging.getLogger("accidents_images")

# -----------------------
# Azure helpers
# -----------------------
def make_blob_service_client(azure_key: str) -> BlobServiceClient:
    if not azure_key:
        raise ValueError("AZURE_STORAGE_CONN_STR is missing.")

    if azure_key.startswith("DefaultEndpointsProtocol=") or "AccountName=" in azure_key:
        return BlobServiceClient.from_connection_string(azure_key)

    if azure_key.startswith("https://") and "sv=" in azure_key:
        account_url = azure_key.split("?", 1)[0]
        sas_token = azure_key.split("?", 1)[1]
        return BlobServiceClient(account_url=account_url, credential=sas_token)

    if ":" in azure_key and len(azure_key.split(":", 1)[1]) > 8:
        account, key = azure_key.split(":", 1)
        conn_str = (
            f"DefaultEndpointsProtocol=https;AccountName={account};"
            f"AccountKey={key};EndpointSuffix=core.windows.net"
        )
        return BlobServiceClient.from_connection_string(conn_str)

    raise ValueError("Unrecognized AZURE_STORAGE_CONN_STR format.")

def parse_blob_path(blob_path_value: str):
    clean_path = blob_path_value.strip("/")
    if "/" in clean_path:
        container_name, prefix = clean_path.split("/", 1)
    else:
        container_name = clean_path
        prefix = ""
    return container_name, prefix

# --- YOU ARE MISSING THESE 3 LINES ---
blob_service = make_blob_service_client(azure_api_key)
container_name, prefix = parse_blob_path(blob_path)
container_client = blob_service.get_container_client(container_name)
# -------------------------------------

def blob_rel(*parts: str) -> str:
    parts = [p.strip("/") for p in parts if p]
    path = "/".join(parts)
    return f"{prefix}/{path}" if prefix else path

def upload_bytes_to_blob(data: bytes, blob_rel_path: str, content_type: str):
    blob_client = container_client.get_blob_client(blob_rel_path)
    blob_client.upload_blob(
        data,
        overwrite=True,
        content_settings=ContentSettings(content_type=content_type),
    )

def download_blob_text(blob_rel_path: str) -> str:
    blob_client = container_client.get_blob_client(blob_rel_path)
    return blob_client.download_blob().readall().decode("utf-8")

def blob_exists(blob_rel_path: str) -> bool:
    return container_client.get_blob_client(blob_rel_path).exists()

def download_blob_bytes(blob_rel_path: str) -> bytes:
    return container_client.get_blob_client(blob_rel_path).download_blob().readall()

def pano_cache_blob(pano_id: str, angle: int) -> str:
    return blob_rel(pano_cache_folder, f"{pano_id}_{angle}.jpg")

def ensure_pano_cached(pano_id: str) -> None:
    for angle in angles:
        cache_blob = pano_cache_blob(pano_id, angle)

        if blob_exists(cache_blob):
            continue

        img_bytes = fetch_image(pano_id=pano_id, heading=angle)
        upload_bytes_to_blob(img_bytes, cache_blob, content_type="image/jpeg")

def save_cached_image(pano_id: str, angle: int, img_bytes: bytes) -> None:
    os.makedirs(local_cache_folder, exist_ok=True)
    cache_path = os.path.join(local_cache_folder, f"{pano_id}_{angle}.jpg")
    with open(cache_path, "wb") as fh:
        fh.write(img_bytes)

def list_metadata_blobs() -> List[str]:
    # This will correctly build "Accidents/Metadata/"
    search_prefix = f"{prefix}/{metadata_folder}/" if prefix else f"{metadata_folder}/"
    search_prefix = search_prefix.replace("//", "/")

    names = []
    # Use list_blobs() which is the correct method for finding files inside a container folder
    for blob in container_client.list_blobs(name_starts_with=search_prefix):
        if blob.name.endswith(".txt"):
            names.append(blob.name)

    return sorted(names)

# -----------------------
# Street View image API
# -----------------------
IMAGE_BASE = "https://maps.googleapis.com/maps/api/streetview"

def build_image_url(pano_id: str, heading: int) -> str:
    params = {
        "size": image_size,
        "pano": pano_id,
        "heading": heading,
        "pitch": pitch,
        "key": street_view_api_key,
    }
    from urllib.parse import urlencode
    return f"{IMAGE_BASE}?{urlencode(params)}"

def fetch_image(pano_id: str, heading: int, retries: int = max_retries) -> bytes:
    if not street_view_api_key:
        raise ValueError("STREET_VIEW_API_KEY is missing.")

    url = build_image_url(pano_id, heading)
    headers = {"User-Agent": user_agent}
    last_exc = None

    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, headers=headers, timeout=request_timeout)
            if r.status_code == 200 and r.content:
                return r.content
            last_exc = Exception(f"HTTP {r.status_code}: {r.text[:200]}")
        except Exception as e:
            last_exc = e

        time.sleep(1.5 ** attempt)

    raise last_exc

# -----------------------
# Checkpoint
# -----------------------
def load_checkpoint():
    cp_blob = blob_rel("checkpoint", checkpoint_local)

    # 1. Use local checkpoint if it exists
    if os.path.exists(checkpoint_local):
        try:
            with open(checkpoint_local, "r", encoding="utf-8") as fh:
                return json.load(fh)
        except Exception:
            logger.exception("Failed reading local checkpoint.")

    # 2. Otherwise download it from Azure
    try:
        if blob_exists(cp_blob):
            logger.info("Downloading checkpoint from Azure...")
            text = download_blob_text(cp_blob)

            with open(checkpoint_local, "w", encoding="utf-8") as fh:
                fh.write(text)

            return json.loads(text)
    except Exception:
        logger.exception("Failed downloading checkpoint from Azure.")

    # 3. Start fresh if neither exists
    return {"done": {}}

def save_checkpoint(cp):
    with open(checkpoint_local, "w", encoding="utf-8") as fh:
        json.dump(cp, fh, indent=2)

    try:
        cp_blob = blob_rel("checkpoint", checkpoint_local)
        upload_bytes_to_blob(json.dumps(cp, indent=2).encode("utf-8"), cp_blob, "application/json")
    except Exception:
        logger.exception("Failed to upload checkpoint to Azure (continuing).")

# -----------------------
# Metadata parsing
# -----------------------
def parse_metadata_text(text: str) -> Dict[str, Any]:
    return json.loads(text)

def get_row_idx_and_pano_id(md: Dict[str, Any], fallback_name: str) -> tuple[int, str]:
    row_idx = md.get("row_idx")
    pano_id = md.get("pano_id")

    if row_idx is None:
        base = os.path.splitext(os.path.basename(fallback_name))[0]
        try:
            row_idx = int(base)
        except Exception:
            row_idx = base

    if not pano_id:
        md_resp = md.get("metadata_response", {})
        pano_id = md_resp.get("pano_id")

    return row_idx, pano_id


# -----------------------
# Main
# -----------------------
def main():
    if not street_view_api_key:
        raise SystemExit("Missing STREET_VIEW_API_KEY.")

    os.makedirs(local_image_folder, exist_ok=True)

    cp = load_checkpoint()
    done = cp.get("done", {})

    metadata_blobs = list_metadata_blobs()
    if not metadata_blobs:
        raise SystemExit(f"No metadata files found under {metadata_folder}/")

    logger.info("Found %d metadata files.", len(metadata_blobs))

    for meta_blob_name in tqdm(metadata_blobs, desc="processing metadata"):
        try:
            meta_text = download_blob_text(meta_blob_name)
            md = parse_metadata_text(meta_text)
            row_idx, pano_id = get_row_idx_and_pano_id(md, meta_blob_name)
        except Exception as e:
            logger.warning("Skipping unreadable metadata %s: %s", meta_blob_name, e)
            continue

        if not pano_id:
            logger.warning("No pano_id for metadata %s", meta_blob_name)
            continue



        for angle in angles:
            key = f"{row_idx}_{angle}"
            if key in done:
                continue

            filename = f"{row_idx}_{angle}.jpg"
            local_path = os.path.join(local_image_folder, filename)
            image_blob = blob_rel(images_folder, filename)

            cache_blob = pano_cache_blob(pano_id, angle)

            if not blob_exists(cache_blob):
                ensure_pano_cached(pano_id)

            img_bytes = download_blob_bytes(cache_blob)

            try:
                with open(local_path, "wb") as fh:
                    fh.write(img_bytes)
            except Exception as e:
                logger.warning("Failed saving local image %s: %s", local_path, e)
                continue

            try:
                upload_bytes_to_blob(img_bytes, image_blob, content_type="image/jpeg")
            except Exception as e:
                logger.warning("Failed uploading image %s: %s", image_blob, e)
                continue

            done[key] = {
                "row_idx": row_idx,
                "angle": angle,
                "pano_id": pano_id,
                "metadata_blob": meta_blob_name,
                "image_blob": image_blob,
                "saved_at_unix": time.time(),
            }
            cp["done"] = done
            save_checkpoint(cp)

    logger.info("Finished. Downloaded %d row-angle images.", len(done))

if __name__ == "__main__":
    main()